<a href="https://colab.research.google.com/github/ManideepLadi/cs6910_assignment3/blob/manideep/RNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Dakshina Dataset from google


In [1]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers


In [2]:
!wget https://storage.googleapis.com/gresearch/dakshina/dakshina_dataset_v1.0.tar

--2021-04-24 07:24:03--  https://storage.googleapis.com/gresearch/dakshina/dakshina_dataset_v1.0.tar
Resolving storage.googleapis.com (storage.googleapis.com)... 172.217.15.112, 172.217.164.144, 142.250.73.240, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|172.217.15.112|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2008340480 (1.9G) [application/x-tar]
Saving to: ‘dakshina_dataset_v1.0.tar’

dakshina_dataset_v1 100%[===================>]   1.87G  28.5MB/s    in 17s     

2021-04-24 07:24:21 (112 MB/s) - ‘dakshina_dataset_v1.0.tar’ saved [2008340480/2008340480]



In [ ]:
!tar -xvf '/content/dakshina_dataset_v1.0.tar'

Preprocess data

In [ ]:
# import string
 
# # load doc into memory
# def load_doc(filename):
# 	# open the file as read only
# 	file = open(filename, mode='rt', encoding='utf-8')
# 	# read all text
# 	text = file.read()
# 	# close the file
# 	file.close()
# 	return text
 
# # split a loaded document into sentences
# def to_pairs(doc):
# 	lines = doc.strip().split('\n')
# 	pairs = [line.split('\t') for line in  lines]
# 	return pairs


In [4]:

# load dataset
filename = 'dakshina_dataset_v1.0/te/lexicons/te.translit.sampled.train.tsv'
# doc = load_doc(filename)
# # split into english-german pairs
# pairs = to_pairs(doc)

In [ ]:
pairs[0]

['అంకిత', 'amkita', '1']

In [ ]:
# # Vectorize the data.
# input_characters = set()
# target_characters = set()
# for pair in pairs:
#   for char in pair[1]:
#     if pair[1] not in input_characters:
#       input_characters.add(char)
#   for char in pair[0]:
#     if char not in target_characters:
#       target_characters.add(char)

# input_characters = sorted(list(input_characters))
# target_characters = sorted(list(target_characters))
# num_encoder_tokens = len(input_characters)
# num_decoder_tokens = len(target_characters)


# print("Number of unique input tokens:", num_encoder_tokens)
# print("Number of unique output tokens:", num_decoder_tokens)


Number of unique input tokens: 26
Number of unique output tokens: 63


In [60]:
# Vectorize the data.
input_texts = []
target_texts = []
input_characters = set()
target_characters = set()
with open(filename, "r", encoding="utf-8") as f:
    lines = f.read().split("\n")
for line in lines[: len(lines) - 1]:
    target_text,input_text, attestation = line.split("\t")
    # We use "tab" as the "start sequence" character
    # for the targets, and "\n" as "end sequence" character.
    target_text = "\t" + target_text + "\n"
    for i in range(int(attestation)):
      input_texts.append(input_text)
      target_texts.append(target_text)
    for char in input_text:
        if char not in input_characters:
            input_characters.add(char)
    for char in target_text:
        if char not in target_characters:
            target_characters.add(char)

input_characters = sorted(list(input_characters))
target_characters = sorted(list(target_characters))
num_encoder_tokens = len(input_characters)
num_decoder_tokens = len(target_characters)
max_encoder_seq_length = max([len(txt) for txt in input_texts])
max_decoder_seq_length = max([len(txt) for txt in target_texts])

print("Number of samples:", len(input_texts))
print("Number of unique input tokens:", num_encoder_tokens)
print("Number of unique output tokens:", num_decoder_tokens)
print("Max sequence length for inputs:", max_encoder_seq_length)
print("Max sequence length for outputs:", max_decoder_seq_length)

Number of samples: 84680
Number of unique input tokens: 26
Number of unique output tokens: 65
Max sequence length for inputs: 25
Max sequence length for outputs: 22


In [61]:
input_texts[1]

'ankita'

In [62]:
target_texts[1]

'\tఅంకిత\n'

In [63]:
input_characters[3]

'd'

In [64]:
input_token_index = dict([(char, i) for i, char in enumerate(input_characters)])
target_token_index = dict([(char, i) for i, char in enumerate(target_characters)])

encoder_input_data = np.zeros(
    (len(input_texts), max_encoder_seq_length, num_encoder_tokens), dtype="float32"
)
decoder_input_data = np.zeros(
    (len(input_texts), max_decoder_seq_length, num_decoder_tokens), dtype="float32"
)
decoder_target_data = np.zeros(
    (len(input_texts), max_decoder_seq_length, num_decoder_tokens), dtype="float32"
)

for i, (input_text, target_text) in enumerate(zip(input_texts, target_texts)):
    for t, char in enumerate(input_text):
        encoder_input_data[i, t, input_token_index[char]] = 1.0
    for t, char in enumerate(target_text):
        # decoder_target_data is ahead of decoder_input_data by one timestep
        decoder_input_data[i, t, target_token_index[char]] = 1.0
        if t > 0:
            # decoder_target_data will be ahead by one timestep
            # and will not include the start character.
            decoder_target_data[i, t - 1, target_token_index[char]] = 1.0

In [65]:
target_texts[0]

'\tఅంకిత\n'

In [66]:
encoder_input_data[0,6]

array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0.], dtype=float32)

In [67]:
decoder_input_data[0,6]

array([0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
      dtype=float32)

In [68]:
# Define an input sequence and process it.
encoder_inputs = keras.Input(shape=(None, num_encoder_tokens))
encoder = keras.layers.LSTM(256, return_state=True)
encoder_outputs, state_h, state_c = encoder(encoder_inputs)

# We discard `encoder_outputs` and only keep the states.
encoder_states = [state_h, state_c]

# Set up the decoder, using `encoder_states` as initial state.
decoder_inputs = keras.Input(shape=(None, num_decoder_tokens))

# We set up our decoder to return full output sequences,
# and to return internal states as well. We don't use the
# return states in the training model, but we will use them in inference.
decoder_lstm = keras.layers.LSTM(256, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(decoder_inputs, initial_state=encoder_states)
decoder_dense = keras.layers.Dense(num_decoder_tokens, activation="softmax")
decoder_outputs = decoder_dense(decoder_outputs)

# Define the model that will turn
# `encoder_input_data` & `decoder_input_data` into `decoder_target_data`
model = keras.Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.summary()

Model: "model_9"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_7 (InputLayer)            [(None, None, 26)]   0                                            
__________________________________________________________________________________________________
input_8 (InputLayer)            [(None, None, 65)]   0                                            
__________________________________________________________________________________________________
lstm_4 (LSTM)                   [(None, 256), (None, 289792      input_7[0][0]                    
__________________________________________________________________________________________________
lstm_5 (LSTM)                   [(None, None, 256),  329728      input_8[0][0]                    
                                                                 lstm_4[0][1]               

In [69]:
model.compile(
    optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"]
)
model.fit(
    [encoder_input_data, decoder_input_data],
    decoder_target_data,
    batch_size=64,
    epochs=100,
    validation_split=0.2
)
# Save model
model.save("s2s")

Epoch 1/100
1059/1059 [==============================] - 13s 10ms/step - loss: 1.2505 - accuracy: 0.0819 - val_loss: 1.2096 - val_accuracy: 0.1245
Epoch 2/100
1059/1059 [==============================] - 9s 9ms/step - loss: 0.8835 - accuracy: 0.1738 - val_loss: 0.9822 - val_accuracy: 0.1734
Epoch 3/100
1059/1059 [==============================] - 9s 9ms/step - loss: 0.6509 - accuracy: 0.2379 - val_loss: 0.8090 - val_accuracy: 0.2212
Epoch 4/100
1059/1059 [==============================] - 9s 9ms/step - loss: 0.5374 - accuracy: 0.2721 - val_loss: 0.7230 - val_accuracy: 0.2316
Epoch 5/100
1059/1059 [==============================] - 9s 9ms/step - loss: 0.4663 - accuracy: 0.2899 - val_loss: 0.6621 - val_accuracy: 0.2479
Epoch 6/100
1059/1059 [==============================] - 10s 9ms/step - loss: 0.3949 - accuracy: 0.3078 - val_loss: 0.6309 - val_accuracy: 0.2542
Epoch 7/100
1059/1059 [==============================] - 9s 9ms/step - loss: 0.3532 - accuracy: 0.3174 - val_loss: 0.6412 - val

INFO:tensorflow:Assets written to: s2s/assets


INFO:tensorflow:Assets written to: s2s/assets


In [70]:
model = keras.models.load_model("s2s")

encoder_inputs = model.input[0]  # input_1
encoder_outputs, state_h_enc, state_c_enc = model.layers[2].output  # lstm_1
encoder_states = [state_h_enc, state_c_enc]
encoder_model = keras.Model(encoder_inputs, encoder_states)

decoder_inputs = model.input[1]  # input_2
decoder_state_input_h = keras.Input(shape=(256,), name="input_3")
decoder_state_input_c = keras.Input(shape=(256,), name="input_4")
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]
decoder_lstm = model.layers[3]
decoder_outputs, state_h_dec, state_c_dec = decoder_lstm(
    decoder_inputs, initial_state=decoder_states_inputs
)
decoder_states = [state_h_dec, state_c_dec]
decoder_dense = model.layers[4]
decoder_outputs = decoder_dense(decoder_outputs)
decoder_model = keras.Model(
    [decoder_inputs] + decoder_states_inputs, [decoder_outputs] + decoder_states
)

# Reverse-lookup token index to decode sequences back to
# something readable.
reverse_input_char_index = dict((i, char) for char, i in input_token_index.items())
reverse_target_char_index = dict((i, char) for char, i in target_token_index.items())


def decode_sequence(input_seq):
    # Encode the input as state vectors.
    states_value = encoder_model.predict(input_seq)

    # Generate empty target sequence of length 1.
    target_seq = np.zeros((1, 1, num_decoder_tokens))

    # Sampling loop for a batch of sequences
    # (to simplify, here we assume a batch of size 1).
    stop_condition = False
    decoded_sentence = ""
    while not stop_condition:
        output_tokens, h, c = decoder_model.predict([target_seq] + states_value)

        # Sample a token
        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_char = reverse_target_char_index[sampled_token_index]
        decoded_sentence += sampled_char

        # Exit condition: either hit max length
        # or find stop character.
        if sampled_char == "\n" or len(decoded_sentence) > max_decoder_seq_length:
            stop_condition = True

        # Update the target sequence (of length 1).
        target_seq = np.zeros((1, 1, num_decoder_tokens))
        target_seq[0, 0, sampled_token_index] = 1.0

        # Update states
        states_value = [h, c]
    return decoded_sentence

In [71]:
for seq_index in range(200):
    # Take one sequence (part of the training set)
    # for trying out decoding.
    input_seq = encoder_input_data[seq_index +100: seq_index + 101]
    decoded_sentence = decode_sequence(input_seq)
    print("-")
    print("Input sentence:", input_texts[seq_index])
    print("Decoded sentence:", decoded_sentence)

-
Input sentence: amkita
Decoded sentence: అంచనాలతో

-
Input sentence: ankita
Decoded sentence: అంచనాలను

-
Input sentence: ankita
Decoded sentence: అంచనాలను

-
Input sentence: ankitha
Decoded sentence: అంచనాలను

-
Input sentence: ankitha
Decoded sentence: అంచనాలు

-
Input sentence: ankitam
Decoded sentence: అంచనాలు

-
Input sentence: ankitham
Decoded sentence: అంచనాలు

-
Input sentence: ankitham
Decoded sentence: అంచనాలు

-
Input sentence: ankitabaavam
Decoded sentence: అంచు

-
Input sentence: ankithabhavam
Decoded sentence: అంచు

-
Input sentence: ankithabhavam
Decoded sentence: అంచు

-
Input sentence: ankatamicchaadu
Decoded sentence: అంచున

-
Input sentence: ankitamicchadu
Decoded sentence: అంచున

-
Input sentence: ankitamichhaadu
Decoded sentence: అంచున

-
Input sentence: ankitamichhaadu
Decoded sentence: అంచుని

-
Input sentence: ankithamicchaadu
Decoded sentence: అంచుని

-
Input sentence: ankithamichhaadu
Decoded sentence: అంచుని

-
Input sentence: amkithamichaaru
Decoded senten

In [ ]:
!pip install wandb -qqq
import wandb
wandb.login()

In [ ]:
sweep_config = {
  'name': 'RNN',
  'method': 'grid',
  'metric': {
      'name': 'accuracy',
      'goal': 'maximize'   
    },
  'parameters': {
        'input_embedding_size': {
            'values': [16, 32, 64, 256]
        },
        'encoder_layers':{
            'values':[1,2,3]
        },
        'decoder_layers':{
            'values':[1,2,3]
        },
        'hidden_layer_size':{
            'values':[16, 32, 64, 256]
        },
        'cell_type':{
            'values':['RNN', 'GRU', 'LSTM']
        },
        'dropout':{
            'values':[0.3,0.2]
        },
        'beam_sizes':{
            'values':['No','Yes']
        }

    }
}

sweep_id = wandb.sweep(sweep_config, project='RNN', entity='manideepladi')